In [15]:

# Difference: the model predicts a full Normal distribution N(mu, sigma) per
# row instead of a single number, via the xgboost-distribution package.

VERSION = 3

import os
import json
import numpy as np
import pandas as pd
import xgboost as xgb
from xgboost_distribution import XGBDistribution

from sklearn.model_selection import (train_test_split, GroupKFold, GroupShuffleSplit)
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from scipy.stats import pearsonr, spearmanr, norm

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

import joblib

import warnings
warnings.filterwarnings("ignore")

DATA_PATH = os.environ.get(
    "SUPPRESSION_CSV",
    r"/Users/PrathamShah/Desktop/Tubingen /Codes/Final_Dataset/community_index_final.csv"
)
OUTDIR = os.environ.get("/Users/PrathamShah/Desktop/Tubingen /Codes/XGBoost_Distribution", ".")
os.makedirs(OUTDIR, exist_ok=True)



df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Target stats -- mean: {df['community_suppression'].mean():.3f}  "
      f"std: {df['community_suppression'].std():.3f}  "
      f"range: [{df['community_suppression'].min():.3f}, {df['community_suppression'].max():.3f}]")



# 41 features (20 R0 + 20 active + diversity_level).

r0_cols       = [f"R0_c{k}" for k in range(1, 21)]
active_cols   = [f"active_c{k}" for k in range(1, 21)]
diversity_col = ['diversity_level']

feature_cols = r0_cols + active_cols + diversity_col
version_label = "Pure nutrients + diversity (no species, no size)"

print(f"Feature count:          {len(feature_cols)}")
print(f"  R0 concentrations:    {len(r0_cols)}")
print(f"  Active flags:         {len(active_cols)}")
print(f"  diversity_level:      {len(diversity_col)}")

missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise KeyError(f"{len(missing)} missing columns: {missing[:5]}")
print("\nAll feature columns confirmed present in file.")

X = df[feature_cols]
y = df['community_suppression']
groups = df["broth_id"].values

print(f"\nX shape: {X.shape}    y shape: {y.shape}")


Dataset shape: 75,000 rows x 168 columns
Target stats -- mean: 0.517  std: 0.333  range: [0.000, 0.999]
Feature count:          41
  R0 concentrations:    20
  Active flags:         20
  diversity_level:      1

All feature columns confirmed present in file.

X shape: (75000, 41)    y shape: (75000,)


In [16]:

strat_key = df["community_size"].astype(str) + "_" + df["diversity_level"].astype(str)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=strat_key)

groups_train = df.loc[X_train.index, "broth_id"].values


def inner_holdout(X_tr, y_tr, groups_tr, frac=0.15, seed=42):
    """Carve a group-aware validation set out of training data for early stopping."""
    gss = GroupShuffleSplit(n_splits=1, test_size=frac, random_state=seed)
    fit_idx, val_idx = next(gss.split(X_tr, y_tr, groups=groups_tr))
    return (X_tr.iloc[fit_idx], y_tr.iloc[fit_idx],
            X_tr.iloc[val_idx], y_tr.iloc[val_idx])


print(f"Train: {X_train.shape[0]:,}   Test: {X_test.shape[0]:,}")


Train: 60,000   Test: 15,000


In [17]:

# uncertainity metrics (used for all the 3 models)

EPS = 1e-6


def gaussian_nll(y_true, mu, sigma):
    """Mean negative log-likelihood of y under N(mu, sigma). Lower is better."""
    sigma = np.maximum(np.asarray(sigma, dtype=float), EPS)
    y_true = np.asarray(y_true, dtype=float)
    mu = np.asarray(mu, dtype=float)
    return float(np.mean(0.5 * np.log(2 * np.pi * sigma ** 2)
                         + (y_true - mu) ** 2 / (2 * sigma ** 2)))


def gaussian_crps(y_true, mu, sigma):
    """Continuous Ranked Probability Score, closed form for a Normal. Lower is better."""
    sigma = np.maximum(np.asarray(sigma, dtype=float), EPS)
    y_true = np.asarray(y_true, dtype=float)
    mu = np.asarray(mu, dtype=float)
    z = (y_true - mu) / sigma
    return float(np.mean(sigma * (z * (2 * norm.cdf(z) - 1)
                                  + 2 * norm.pdf(z) - 1.0 / np.sqrt(np.pi))))


def coverage_at(y_true, mu, sigma, level):
    """Fraction of true values falling inside the central `level` interval."""
    sigma = np.maximum(np.asarray(sigma, dtype=float), EPS)
    z = norm.ppf(0.5 + level / 2.0)
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(mu)) <= z * sigma))


def mean_interval_width(mu, sigma, level=0.95):
    """Average width of the central `level` interval. Lower is sharper."""
    z = norm.ppf(0.5 + level / 2.0)
    return float(np.mean(2 * z * np.maximum(np.asarray(sigma, dtype=float), EPS)))


def pit_values(y_true, mu, sigma):
    """Probability Integral Transform. Perfectly calibrated -> Uniform(0,1)."""
    sigma = np.maximum(np.asarray(sigma, dtype=float), EPS)
    return norm.cdf((np.asarray(y_true) - np.asarray(mu)) / sigma)


def pit_ks_stat(y_true, mu, sigma):
    """Kolmogorov-Smirnov distance between the PIT sample and Uniform(0,1)."""
    p = np.sort(pit_values(y_true, mu, sigma))
    n = len(p)
    grid = np.arange(1, n + 1) / n
    return float(np.max(np.abs(p - grid)))


def sigma_error_spearman(y_true, mu, sigma):
    """Does a larger predicted sigma actually mean a larger error? Want > 0."""
    err = np.abs(np.asarray(y_true) - np.asarray(mu))
    return float(spearmanr(sigma, err).statistic)


def calibration_factor(y_true, mu, sigma, level=0.95):
    """Scalar s such that mu +/- z*s*sigma attains `level` empirical coverage.

    This is the one number the optimizer needs most: an uncalibrated sigma makes
    an acquisition function either far too underconfident or overconfident.
    """
    sigma = np.maximum(np.asarray(sigma, dtype=float), EPS)
    z_abs = np.abs(np.asarray(y_true) - np.asarray(mu)) / sigma
    return float(np.quantile(z_abs, level) / norm.ppf(0.5 + level / 2.0))


def uncertainty_report(y_true, mu, sigma, label=""):
    
    rep = {
        'r2':        r2_score(y_true, mu),
        'mae':       mean_absolute_error(y_true, mu),
        'rmse':      float(np.sqrt(mean_squared_error(y_true, mu))),
        'nll':       gaussian_nll(y_true, mu, sigma),
        'crps':      gaussian_crps(y_true, mu, sigma),
        'cov50':     coverage_at(y_true, mu, sigma, 0.50),
        'cov80':     coverage_at(y_true, mu, sigma, 0.80),
        'cov90':     coverage_at(y_true, mu, sigma, 0.90),
        'cov95':     coverage_at(y_true, mu, sigma, 0.95),
        'width95':   mean_interval_width(mu, sigma, 0.95),
        'mean_sigma': float(np.mean(sigma)),
        'pit_ks':    pit_ks_stat(y_true, mu, sigma),
        'sig_err_rho': sigma_error_spearman(y_true, mu, sigma),
        'calib_factor': calibration_factor(y_true, mu, sigma, 0.95),
    }
    if label:
        print(f"\nUncertainty scorecard: {label}")
        print(f"  R2   = {rep['r2']:.4f}    MAE = {rep['mae']:.4f}    RMSE = {rep['rmse']:.4f}")
        print(f"  NLL  = {rep['nll']:.4f}   CRPS = {rep['crps']:.4f}")
        print(f"  Coverage  50% = {rep['cov50']:.3f}   80% = {rep['cov80']:.3f}   "
              f"90% = {rep['cov90']:.3f}   95% = {rep['cov95']:.3f}")
        print(f"  Mean sigma = {rep['mean_sigma']:.4f}   mean 95% width = {rep['width95']:.4f}")
        print(f"  PIT KS distance = {rep['pit_ks']:.4f}  (0 = perfectly calibrated)")
        print(f"  Spearman(sigma, |error|) = {rep['sig_err_rho']:.4f}  (> 0 = sigma is informative)")
        print(f"  Calibration factor for 95% = {rep['calib_factor']:.4f}  "
              f"(1.0 = already calibrated, > 1 = overconfident)")
    return rep



In [18]:

# initial model

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


Xi_fit, yi_fit, Xi_val, yi_val = inner_holdout(X_train, y_train, groups_train)

model_init = XGBDistribution(
    distribution="normal",
    natural_gradient=True,
    n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, min_child_weight=3, reg_lambda=1.0,
    random_state=42, n_jobs=-1, early_stopping_rounds=30, verbosity=0)

model_init.fit(Xi_fit, yi_fit, eval_set=[(Xi_val, yi_val)], verbose=False)

pred_init = model_init.predict(X_test)          
mu_init, sd_init = pred_init.loc, pred_init.scale
print(f"\nXGBDistribution (initial):  R2 = {r2_score(y_test, mu_init):.4f}  "
      f"MAE = {mean_absolute_error(y_test, mu_init):.4f}  "
      f"NLL = {gaussian_nll(y_test, mu_init, sd_init):.4f}  "
      f"trees used = {model_init.best_iteration}")




XGBDistribution (initial):  R2 = 0.6209  MAE = 0.1638  NLL = -0.2240  trees used = 55


In [19]:

# hyperparameter tuning
# The loop below is the group-aware replacement, scored on validation NLL,
# which is the correct criterion for a distributional model.

RUN_TUNING = False   # set True to actually search; here I am just using previous best params from XGB03

param_grid = {
    "max_depth":        [5],
    "learning_rate":    [0.1],
    "subsample":        [0.6],
    "colsample_bytree": [0.8],
}
# Full grid to use when RUN_TUNING = True:
# param_grid = {"max_depth": [4, 5, 6, 8], "learning_rate": [0.03, 0.05, 0.1],
#               "subsample": [0.6, 0.8, 1.0], "colsample_bytree": [0.6, 0.8, 1.0]}

from itertools import product


def grid_search_nll(X_tr, y_tr, groups_tr, grid, n_splits=3, n_estimators=300):
    """Group-aware grid search scored on held-out-broth Gaussian NLL."""
    keys = list(grid.keys())
    combos = [dict(zip(keys, vals)) for vals in product(*[grid[k] for k in keys])]
    gkf = GroupKFold(n_splits=n_splits)
    rows = []
    print(f"Searching {len(combos)} candidates x {n_splits} group folds "
          f"= {len(combos) * n_splits} fits")
    for i, params in enumerate(combos, start=1):
        fold_nll = []
        for tr_idx, va_idx in gkf.split(X_tr, y_tr, groups=groups_tr):
            Xa, ya = X_tr.iloc[tr_idx], y_tr.iloc[tr_idx]
            Xb, yb = X_tr.iloc[va_idx], y_tr.iloc[va_idx]
            m = XGBDistribution(distribution="normal", n_estimators=n_estimators,
                                min_child_weight=3, reg_lambda=1.0, random_state=42,
                                n_jobs=-1, verbosity=0, **params)
            m.fit(Xa, ya, verbose=False)      # no eval_set: nothing to peek at
            p = m.predict(Xb)
            fold_nll.append(gaussian_nll(yb, p.loc, p.scale))
        rows.append({**params, 'mean_nll': float(np.mean(fold_nll)),
                     'std_nll': float(np.std(fold_nll))})
        print(f"  [{i}/{len(combos)}] {params} -> NLL {np.mean(fold_nll):.4f}")
    res = pd.DataFrame(rows).sort_values('mean_nll').reset_index(drop=True)
    return res


if RUN_TUNING:
    tune_df = grid_search_nll(X_train, y_train, groups_train, param_grid)
    best_params = {k: tune_df.loc[0, k] for k in param_grid.keys()}
    print(f"\nBest params by held-out-broth NLL: {best_params}")
    print(f"Best NLL: {tune_df.loc[0, 'mean_nll']:.4f}")
    tune_df.to_csv(os.path.join(OUTDIR, "xgbd03_tuning.csv"), index=False)
else:
    best_params = {k: v[0] for k, v in param_grid.items()}
    print(f"Reusing known-best params (tuning skipped): {best_params}")


Reusing known-best params (tuning skipped): {'max_depth': 5, 'learning_rate': 0.1, 'subsample': 0.6, 'colsample_bytree': 0.8}


In [20]:


# final model
Xf_fit, yf_fit, Xf_val, yf_val = inner_holdout(X_train, y_train, groups_train)

model_final = XGBDistribution(
    distribution="normal", natural_gradient=True,
    **best_params, n_estimators=1000, min_child_weight=3, reg_lambda=1.0,
    random_state=42, n_jobs=-1, early_stopping_rounds=40, verbosity=0)

model_final.fit(Xf_fit, yf_fit,
                eval_set=[(Xf_fit, yf_fit), (Xf_val, yf_val)], verbose=False)

METRIC_KEY = "NormalDistribution-NLL"
evals = model_final.evals_result()
tr_curve = np.asarray(evals['validation_0'][METRIC_KEY])
va_curve = np.asarray(evals['validation_1'][METRIC_KEY])
best_it = model_final.best_iteration

fig_tc, ax_tc = plt.subplots(figsize=(9, 4))
ax_tc.plot(range(1, len(tr_curve) + 1), tr_curve, color='steelblue', lw=1.5, label='Train NLL')
ax_tc.plot(range(1, len(va_curve) + 1), va_curve, color='darkorange', lw=1.5, label='Validation NLL')
ax_tc.axvline(best_it + 1, color='red', linestyle='--', lw=1.2, label=f'Best iteration = {best_it + 1}')
ax_tc.set_xlabel('Boosting round')
ax_tc.set_ylabel('Negative log-likelihood')
ax_tc.set_title(f'XGBDistribution Training Curve -- best iter={best_it + 1}, '
                f'val NLL={va_curve[best_it]:.4f}')
ax_tc.legend(fontsize=9)
ax_tc.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'xgbd03_training_curve.png'), dpi=150, bbox_inches='tight')
plt.close()
print("Saved: xgbd03_training_curve.png")

pred_final = model_final.predict(X_test)
mu_final, sd_final = pred_final.loc, pred_final.scale

r2_final = r2_score(y_test, mu_final)
mae_final = mean_absolute_error(y_test, mu_final)
rmse_final = float(np.sqrt(mean_squared_error(y_test, mu_final)))

print(f"\nFinal (interpolation reference): R2 = {r2_final:.4f}  MAE = {mae_final:.4f}  "
      f"RMSE = {rmse_final:.4f}  trees = {best_it}")
rep_interp = uncertainty_report(y_test, mu_final, sd_final, "interpolation test set")


Saved: xgbd03_training_curve.png

Final (interpolation reference): R2 = 0.6643  MAE = 0.1394  RMSE = 0.1926  trees = 319

Uncertainty scorecard: interpolation test set
  R2   = 0.6643    MAE = 0.1394    RMSE = 0.1926
  NLL  = -0.7005   CRPS = 0.0979
  Coverage  50% = 0.470   80% = 0.804   90% = 0.907   95% = 0.949
  Mean sigma = 0.1611   mean 95% width = 0.6317
  PIT KS distance = 0.0627  (0 = perfectly calibrated)
  Spearman(sigma, |error|) = 0.5673  (> 0 = sigma is informative)
  Calibration factor for 95% = 1.0069  (1.0 = already calibrated, > 1 = overconfident)


In [21]:


# GroupKFold: held-out broths + OOF sigma
gkf = GroupKFold(n_splits=5)

fold_rows = []
oof_mu = np.full(len(X), np.nan)
oof_sd = np.full(len(X), np.nan)

print("\nGroupKFold evaluation (true held-out broths):")

for fold_num, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]
    groups_tr = groups[tr_idx]

    assert len(set(groups_tr).intersection(set(groups[te_idx]))) == 0

    Xg_fit, yg_fit, Xg_val, yg_val = inner_holdout(X_tr, y_tr, groups_tr,
                                                   frac=0.15, seed=42 + fold_num)

    m_fold = XGBDistribution(
        distribution="normal", natural_gradient=True,
        **best_params, n_estimators=1000, min_child_weight=3, reg_lambda=1.0,
        random_state=42, n_jobs=-1, early_stopping_rounds=40, verbosity=0)
    m_fold.fit(Xg_fit, yg_fit, eval_set=[(Xg_val, yg_val)], verbose=False)

    p_fold = m_fold.predict(X_te)
    oof_mu[te_idx] = p_fold.loc
    oof_sd[te_idx] = p_fold.scale

    row = {
        'fold': fold_num,
        'n_test': len(X_te),
        'r2': r2_score(y_te, p_fold.loc),
        'mae': mean_absolute_error(y_te, p_fold.loc),
        'rmse': float(np.sqrt(mean_squared_error(y_te, p_fold.loc))),
        'pearson': pearsonr(y_te, p_fold.loc).statistic,
        'spearman': spearmanr(y_te, p_fold.loc).statistic,
        'nll': gaussian_nll(y_te, p_fold.loc, p_fold.scale),
        'crps': gaussian_crps(y_te, p_fold.loc, p_fold.scale),
        'cov95': coverage_at(y_te, p_fold.loc, p_fold.scale, 0.95),
        'mean_sigma': float(np.mean(p_fold.scale)),
        'trees': m_fold.best_iteration,
    }
    fold_rows.append(row)
    print(f"  Fold {fold_num}: R2={row['r2']:.4f}  MAE={row['mae']:.4f}  "
          f"NLL={row['nll']:.4f}  cov95={row['cov95']:.3f}  "
          f"mean_sigma={row['mean_sigma']:.4f}  trees={row['trees']}")

folds_df = pd.DataFrame(fold_rows)

# Pooled out-of-fold metrics. Pooling, not averaging per-fold R2, because
# per-fold R2 is unstable when a fold has low variance in y.
print(f"\nPOOLED out-of-fold (all {len(X):,} rows, every row predicted by a "
      f"model that never saw its broth):")
rep_oof = uncertainty_report(y, oof_mu, oof_sd, "pooled held-out broths")

print(f"\nInterpolation R2 = {r2_final:.4f}   Held-out-broth R2 = {rep_oof['r2']:.4f}   "
      f"Gap = {r2_final - rep_oof['r2']:.4f}")
print(f"Interpolation mean sigma = {rep_interp['mean_sigma']:.4f}   "
      f"Held-out-broth mean sigma = {rep_oof['mean_sigma']:.4f}   "
      f"Ratio = {rep_oof['mean_sigma'] / max(rep_interp['mean_sigma'], EPS):.3f}")
print("A ratio well above 1.0 means the model widens its intervals on unseen")
print("broths on its own. A ratio near 1.0 means it does not, and the optimizer")
print("must rely on the calibration factor computed below instead.")





GroupKFold evaluation (true held-out broths):
  Fold 1: R2=0.5847  MAE=0.1553  NLL=0.7134  cov95=0.890  mean_sigma=0.1684  trees=347
  Fold 2: R2=0.2069  MAE=0.2224  NLL=0.2060  cov95=0.789  mean_sigma=0.1763  trees=235
  Fold 3: R2=0.4420  MAE=0.2103  NLL=0.0123  cov95=0.915  mean_sigma=0.2167  trees=27
  Fold 4: R2=0.4498  MAE=0.2048  NLL=-0.0165  cov95=0.921  mean_sigma=0.2065  trees=34
  Fold 5: R2=0.4779  MAE=0.1907  NLL=-0.0712  cov95=0.957  mean_sigma=0.2028  trees=36

POOLED out-of-fold (all 75,000 rows, every row predicted by a model that never saw its broth):

Uncertainty scorecard: pooled held-out broths
  R2   = 0.4338    MAE = 0.1967    RMSE = 0.2504
  NLL  = 0.1688   CRPS = 0.1382
  Coverage  50% = 0.409   80% = 0.693   90% = 0.812   95% = 0.894
  Mean sigma = 0.1941   mean 95% width = 0.7610
  PIT KS distance = 0.0742  (0 = perfectly calibrated)
  Spearman(sigma, |error|) = 0.3895  (> 0 = sigma is informative)
  Calibration factor for 95% = 1.1991  (1.0 = already calibr

In [22]:

# GroupShuffleSplit repeatability
gss20 = GroupShuffleSplit(n_splits=20, test_size=0.20, random_state=42)
gss_rows = []

for split_num, (tr_idx, te_idx) in enumerate(gss20.split(X, y, groups=groups), start=1):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]
    groups_tr = groups[tr_idx]
    assert len(set(groups_tr).intersection(set(groups[te_idx]))) == 0

    Xg_fit, yg_fit, Xg_val, yg_val = inner_holdout(X_tr, y_tr, groups_tr,
                                                   frac=0.15, seed=42 + split_num)
    m = XGBDistribution(
        distribution="normal", natural_gradient=True,
        **best_params, n_estimators=1000, min_child_weight=3, reg_lambda=1.0,
        random_state=42 + split_num, n_jobs=-1, early_stopping_rounds=40, verbosity=0)
    m.fit(Xg_fit, yg_fit, eval_set=[(Xg_val, yg_val)], verbose=False)
    p = m.predict(X_te)

    gss_rows.append({
        'split': split_num,
        'r2': r2_score(y_te, p.loc),
        'mae': mean_absolute_error(y_te, p.loc),
        'rmse': float(np.sqrt(mean_squared_error(y_te, p.loc))),
        'nll': gaussian_nll(y_te, p.loc, p.scale),
        'cov95': coverage_at(y_te, p.loc, p.scale, 0.95),
        'mean_sigma': float(np.mean(p.scale)),
        'calib': calibration_factor(y_te, p.loc, p.scale, 0.95),
    })
    print(f"Split {split_num:02d}: R2={gss_rows[-1]['r2']:.4f}  "
          f"NLL={gss_rows[-1]['nll']:.4f}  cov95={gss_rows[-1]['cov95']:.3f}  "
          f"calib={gss_rows[-1]['calib']:.3f}")

gss_df = pd.DataFrame(gss_rows)
print("\nGroupShuffleSplit held-out-broth summary:")
for c in ['r2', 'mae', 'rmse', 'nll', 'cov95', 'mean_sigma', 'calib']:
    print(f"  {c:<11} mean={gss_df[c].mean():.4f}  std={gss_df[c].std():.4f}")



Split 01: R2=0.1153  NLL=0.2890  cov95=0.857  calib=1.090
Split 02: R2=0.5552  NLL=5.9189  cov95=0.867  calib=1.497
Split 03: R2=0.3510  NLL=0.1794  cov95=0.818  calib=1.554
Split 04: R2=0.3880  NLL=0.1977  cov95=0.811  calib=1.451
Split 05: R2=0.4093  NLL=-0.0198  cov95=0.871  calib=1.222
Split 06: R2=0.4883  NLL=0.0304  cov95=0.891  calib=1.299
Split 07: R2=0.4029  NLL=0.0349  cov95=0.985  calib=0.846
Split 08: R2=0.3055  NLL=0.0620  cov95=0.928  calib=1.117
Split 09: R2=0.4328  NLL=-0.0252  cov95=0.968  calib=0.917
Split 10: R2=0.2570  NLL=0.1207  cov95=0.988  calib=0.954
Split 11: R2=0.3995  NLL=-0.0118  cov95=0.874  calib=1.481
Split 12: R2=0.3941  NLL=0.0504  cov95=0.965  calib=0.952
Split 13: R2=0.4034  NLL=0.0974  cov95=0.890  calib=1.234
Split 14: R2=0.2319  NLL=1.4120  cov95=0.714  calib=2.522
Split 15: R2=0.3938  NLL=0.0717  cov95=0.913  calib=1.232
Split 16: R2=0.6106  NLL=-0.1295  cov95=0.978  calib=0.845
Split 17: R2=0.3698  NLL=0.1358  cov95=0.895  calib=1.305
Split 18: 

In [23]:


# per-condition breakdown (with uncertainty)
print("Performance by diversity level and community size")

results_df = pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          mu_final,
    'y_sigma':         sd_final,
    'residual':        y_test.values - mu_final,
    'diversity_level': df.loc[X_test.index, 'diversity_level'].values,
    'community_size':  df.loc[X_test.index, 'community_size'].values,
})


def condition_metrics(g):
    y_t = g['y_true'].values
    y_p = g['y_pred'].values
    s_p = g['y_sigma'].values
    return {
        'n': len(y_t),
        'r2': r2_score(y_t, y_p),
        'mae': mean_absolute_error(y_t, y_p),
        'rmse': float(np.sqrt(mean_squared_error(y_t, y_p))),
        'spearman': spearmanr(y_t, y_p).statistic,
        'bias': float(np.mean(y_p - y_t)),
        'std_true': float(np.std(y_t)),
        'sigma': float(np.mean(s_p)),
        'nll': gaussian_nll(y_t, y_p, s_p),
        'cov95': coverage_at(y_t, y_p, s_p, 0.95),
    }


hdr = (f"{'':6} {'R2':>7} {'MAE':>7} {'RMSE':>7} {'Spear':>7} {'Bias':>8} "
       f"{'std(y)':>7} {'sigma':>7} {'NLL':>8} {'cov95':>7} {'N':>6}")

print("\nBy diversity level:")
print(hdr)
print("-" * len(hdr))
for d, grp in results_df.groupby('diversity_level'):
    m = condition_metrics(grp)
    flag = "  *" if m['std_true'] < 0.08 else "   "
    print(f"D={d:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['spearman']:>7.4f} {m['bias']:>8.4f} {m['std_true']:>7.4f} "
          f"{m['sigma']:>7.4f} {m['nll']:>8.4f} {m['cov95']:>7.3f} {m['n']:>6,}{flag}")
print("  * std(y) < 0.08: R2 is unreliable here. Read MAE, Spearman and NLL instead.")

print("\nBy community size:")
print(hdr)
print("-" * len(hdr))
for n, grp in results_df.groupby('community_size'):
    m = condition_metrics(grp)
    print(f"N={n:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['spearman']:>7.4f} {m['bias']:>8.4f} {m['std_true']:>7.4f} "
          f"{m['sigma']:>7.4f} {m['nll']:>8.4f} {m['cov95']:>7.3f} {m['n']:>6,}")



Performance by diversity level and community size

By diversity level:
            R2     MAE    RMSE   Spear     Bias  std(y)   sigma      NLL   cov95      N
---------------------------------------------------------------------------------------
D=1     0.8492  0.0933  0.1682  0.9375  -0.0008  0.4330  0.1112  -2.1472   0.951  3,000   
D=2     0.5679  0.1644  0.2343  0.7817   0.0235  0.3564  0.1909  -0.3155   0.928  3,000   
D=4     0.3924  0.1867  0.2294  0.6182  -0.0281  0.2944  0.2026  -0.0449   0.937  3,000   
D=8     0.5533  0.1397  0.1738  0.7127  -0.0014  0.2600  0.1672  -0.3944   0.964  3,000   
D=16    0.3445  0.1129  0.1395  0.5620  -0.0050  0.1722  0.1338  -0.6005   0.964  3,000   
  * std(y) < 0.08: R2 is unreliable here. Read MAE, Spearman and NLL instead.

By community size:
            R2     MAE    RMSE   Spear     Bias  std(y)   sigma      NLL   cov95      N
---------------------------------------------------------------------------------------
N=12    0.4617  0.1741  

In [24]:

# calibration factor for the optimizer
# Fitted on POOLED held-out-broth predictions, which is the honest setting:
# the optimizer will always be asked about broths the model has never seen.
CALIB_LEVEL = 0.95
sigma_scale = calibration_factor(y, oof_mu, oof_sd, CALIB_LEVEL)

print(f"\nCalibration on pooled held-out broths (level {CALIB_LEVEL:.0%}):")
print(f"  raw coverage        = {coverage_at(y, oof_mu, oof_sd, CALIB_LEVEL):.4f}")
print(f"  sigma_scale factor  = {sigma_scale:.4f}")
print(f"  calibrated coverage = "
      f"{coverage_at(y, oof_mu, oof_sd * sigma_scale, CALIB_LEVEL):.4f}")
print("Multiply every predicted sigma by sigma_scale before feeding an")
print("acquisition function, otherwise the exploration term is mis-scaled.")




Calibration on pooled held-out broths (level 95%):
  raw coverage        = 0.8943
  sigma_scale factor  = 1.1991
  calibrated coverage = 0.9522
Multiply every predicted sigma by sigma_scale before feeding an
acquisition function, otherwise the exploration term is mis-scaled.


In [28]:


# FIGURE 1: standard panels
output_name = os.path.join(OUTDIR, f"xgbd03_results_v{VERSION}.png")

fig = plt.figure(figsize=(18, 14), dpi=120)
fig.suptitle(
    f'XGBDistribution -- {version_label}\n'
    f'R2={r2_final:.3f}  MAE={mae_final:.3f}  RMSE={rmse_final:.3f}  '
    f'NLL={rep_interp["nll"]:.3f}  CRPS={rep_interp["crps"]:.3f}',
    fontsize=12, fontweight='bold', y=0.99)

ax1 = fig.add_subplot(2, 3, 1)
for d, col in zip([1, 2, 4, 8, 16], ['#e41a1c', '#ff7f00', '#4daf4a', '#377eb8', '#984ea3']):
    mask = results_df['diversity_level'].values == d
    ax1.scatter(y_test.values[mask], mu_final[mask], alpha=0.15, s=2, c=col, label=f'D={d}')
ax1.plot([0, 1], [0, 1], 'r--', lw=1.5)
ax1.set_xlabel('True suppression')
ax1.set_ylabel('Predicted mean')
ax1.set_title(f'Predicted vs Actual  R2={r2_final:.3f}')
ax1.legend(fontsize=7, markerscale=3)

ax2 = fig.add_subplot(2, 3, 2)
resid = results_df['residual'].values
for n, col in zip([12, 16, 24, 32, 48], ['#1b9e77', '#d95f02', '#7570b3', '#e7298a', '#66a61e']):
    mask = results_df['community_size'].values == n
    ax2.scatter(mu_final[mask], resid[mask], alpha=0.15, s=2, c=col, label=f'N={n}')
ax2.axhline(0, color='r', linestyle='--', lw=1.5)
p5, p95 = np.percentile(resid, [5, 95])
ax2.axhline(p5, color='grey', linestyle=':', lw=1, label=f'5th: {p5:.3f}')
ax2.axhline(p95, color='grey', linestyle=':', lw=1, label=f'95th: {p95:.3f}')
ax2.set_xlabel('Predicted mean')
ax2.set_ylabel('Residual (true - predicted)')
ax2.set_title('Residuals (coloured by size -- not a feature)')
ax2.legend(fontsize=7, markerscale=3)

ax3 = fig.add_subplot(2, 3, 3)


def feat_colour(f):
    if f in r0_cols:
        return '#4CAF50'
    if f in active_cols:
        return '#FF9800'
    return '#9C27B0'


ax4 = fig.add_subplot(2, 3, 4)
divs = sorted(results_df['diversity_level'].unique())
r2_div = [r2_score(results_df[results_df['diversity_level'] == d]['y_true'],
                   results_df[results_df['diversity_level'] == d]['y_pred']) for d in divs]
ax4.bar([str(d) for d in divs], r2_div, color='steelblue', edgecolor='navy')
ax4.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R2={r2_final:.3f}')
ax4.set_xlabel('Diversity level')
ax4.set_ylabel('R2')
ax4.set_title('R2 by Diversity Level')
ax4.set_ylim(0, 1.05)
ax4.legend(fontsize=8)

ax5 = fig.add_subplot(2, 3, 5)
sizes = sorted(results_df['community_size'].unique())
r2_sz = [r2_score(results_df[results_df['community_size'] == n]['y_true'],
                  results_df[results_df['community_size'] == n]['y_pred']) for n in sizes]
ax5.bar([str(n) for n in sizes], r2_sz, color='darkorange', edgecolor='saddlebrown')
ax5.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R2={r2_final:.3f}')
ax5.set_xlabel('Community size N (not a feature)')
ax5.set_ylabel('R2')
ax5.set_title('R2 by Community Size')
ax5.set_ylim(0, 1.05)
ax5.legend(fontsize=8)



plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(output_name, dpi=150, bbox_inches='tight')
plt.close()
print(f"\nPlot saved: {output_name}")



# FIGURE 2: uncertainty diagnostics
unc_name = os.path.join(OUTDIR, f"xgbd03_uncertainty_v{VERSION}.png")

fig = plt.figure(figsize=(18, 11), dpi=120)
fig.suptitle(
    f'XGBDistribution -- Uncertainty Diagnostics ({version_label})\n'
    f'test NLL={rep_interp["nll"]:.3f}  CRPS={rep_interp["crps"]:.3f}  '
    f'cov95={rep_interp["cov95"]:.3f}  sigma_scale={sigma_scale:.3f}',
    fontsize=12, fontweight='bold', y=0.99)

# Panel 1: reliability curve, nominal vs empirical coverage
axA = fig.add_subplot(2, 3, 1)
levels = np.linspace(0.05, 0.99, 30)
emp_test = [coverage_at(y_test, mu_final, sd_final, L) for L in levels]
emp_oof = [coverage_at(y, oof_mu, oof_sd, L) for L in levels]
emp_cal = [coverage_at(y, oof_mu, oof_sd * sigma_scale, L) for L in levels]
axA.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect calibration')
axA.plot(levels, emp_test, color='steelblue', lw=2, label='Interpolation test')
axA.plot(levels, emp_oof, color='darkorange', lw=2, label='Held-out broths (raw)')
axA.plot(levels, emp_cal, color='green', lw=2, label='Held-out broths (calibrated)')
axA.set_xlabel('Nominal coverage')
axA.set_ylabel('Empirical coverage')
axA.set_title('Reliability Curve\n(below diagonal = overconfident)')
axA.legend(fontsize=8)
axA.grid(True, alpha=0.3)

# Panel 2: PIT histogram
axB = fig.add_subplot(2, 3, 2)
pit_test = pit_values(y_test, mu_final, sd_final)
pit_oof = pit_values(y, oof_mu, oof_sd * sigma_scale)
axB.hist(pit_test, bins=20, range=(0, 1), alpha=0.55, color='steelblue',
         density=True, label='Interpolation test')
axB.hist(pit_oof, bins=20, range=(0, 1), alpha=0.55, color='green',
         density=True, label='Held-out (calibrated)')
axB.axhline(1.0, color='red', linestyle='--', lw=1.5, label='Uniform target')
axB.set_xlabel('PIT value')
axB.set_ylabel('Density')
axB.set_title(f'PIT Histogram\nKS={rep_interp["pit_ks"]:.3f} (flat = calibrated)')
axB.legend(fontsize=8)

# Panel 3: does sigma predict the error?
axC = fig.add_subplot(2, 3, 3)
abs_err = np.abs(y_test.values - mu_final)
order = np.argsort(sd_final)
n_bins = 20
bin_edges = np.array_split(order, n_bins)
bin_sigma = [sd_final[b].mean() for b in bin_edges]
bin_err = [abs_err[b].mean() for b in bin_edges]
bin_err_sd = [abs_err[b].std() / np.sqrt(len(b)) for b in bin_edges]
axC.errorbar(bin_sigma, bin_err, yerr=bin_err_sd, fmt='o-', color='purple',
             capsize=3, lw=1.5, ms=5)
lims = [min(min(bin_sigma), min(bin_err)), max(max(bin_sigma), max(bin_err))]
axC.plot(lims, lims, 'k--', lw=1, label='y = x')
axC.set_xlabel('Predicted sigma (binned mean)')
axC.set_ylabel('Actual |error| (binned mean)')
axC.set_title(f'Sigma vs Error\nSpearman rho={rep_interp["sig_err_rho"]:.3f}')
axC.legend(fontsize=8)
axC.grid(True, alpha=0.3)

# Panel 4: prediction intervals for a sorted sample of test rows
axD = fig.add_subplot(2, 3, 4)
rng = np.random.default_rng(42)
samp = rng.choice(len(y_test), size=min(300, len(y_test)), replace=False)
srt = samp[np.argsort(mu_final[samp])]
xs = np.arange(len(srt))
z95 = norm.ppf(0.975)
axD.fill_between(xs, mu_final[srt] - z95 * sd_final[srt] * sigma_scale,
                 mu_final[srt] + z95 * sd_final[srt] * sigma_scale,
                 color='steelblue', alpha=0.25, label='95% interval (calibrated)')
axD.plot(xs, mu_final[srt], color='navy', lw=1.2, label='Predicted mean')
inside = np.abs(y_test.values[srt] - mu_final[srt]) <= z95 * sd_final[srt] * sigma_scale
axD.scatter(xs[inside], y_test.values[srt][inside], s=6, c='green', label='True (inside)')
axD.scatter(xs[~inside], y_test.values[srt][~inside], s=10, c='red', label='True (outside)')
axD.set_xlabel('Test rows sorted by predicted mean')
axD.set_ylabel('Suppression')
axD.set_title(f'Prediction Intervals ({inside.mean():.1%} inside)')
axD.legend(fontsize=7, markerscale=1.5)

# Panel 5: sigma distribution by diversity level
axE = fig.add_subplot(2, 3, 5)
data_by_d = [results_df.loc[results_df['diversity_level'] == d, 'y_sigma'].values
             for d in divs]
bp = axE.boxplot(data_by_d, label=[f'D={d}' for d in divs], patch_artist=True,
                 showfliers=False)
for patch, col in zip(bp['boxes'], ['#e41a1c', '#ff7f00', '#4daf4a', '#377eb8', '#984ea3']):
    patch.set_facecolor(col)
    patch.set_alpha(0.6)
axE.set_ylabel('Predicted sigma')
axE.set_title('Uncertainty by Diversity Level\n(where is the model least sure?)')
axE.grid(True, alpha=0.3, axis='y')

# Panel 6: interpolation sigma vs held-out-broth sigma
axF = fig.add_subplot(2, 3, 6)
axF.hist(sd_final, bins=50, alpha=0.55, color='steelblue', density=True,
         label=f'Interpolation (mean={np.mean(sd_final):.3f})')
axF.hist(oof_sd, bins=50, alpha=0.55, color='darkorange', density=True,
         label=f'Held-out broths (mean={np.mean(oof_sd):.3f})')
axF.axvline(np.mean(sd_final), color='steelblue', linestyle='--', lw=1.5)
axF.axvline(np.mean(oof_sd), color='darkorange', linestyle='--', lw=1.5)
axF.set_xlabel('Predicted sigma')
axF.set_ylabel('Density')
axF.set_title('Does Sigma Grow on Unseen Broths?\n'
              f'ratio={np.mean(oof_sd) / max(np.mean(sd_final), EPS):.2f}')
axF.legend(fontsize=8)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(unc_name, dpi=150, bbox_inches='tight')
plt.close()
print(f"Plot saved: {unc_name}")




Plot saved: ./xgbd03_results_v3.png
Plot saved: ./xgbd03_uncertainty_v3.png


In [29]:


# save model, predictions and BO bundle
model_path = os.path.join(OUTDIR, f"xgbd03_v{VERSION}_model.joblib")
joblib.dump(model_final, model_path)

pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          mu_final,
    'y_sigma':         sd_final,
    'y_sigma_calib':   sd_final * sigma_scale,
    'lo95':            mu_final - norm.ppf(0.975) * sd_final * sigma_scale,
    'hi95':            mu_final + norm.ppf(0.975) * sd_final * sigma_scale,
    'residual':        y_test.values - mu_final,
    'diversity_level': results_df['diversity_level'].values,
    'community_size':  results_df['community_size'].values,
}).to_csv(os.path.join(OUTDIR, f"xgbd03_v{VERSION}_predictions.csv"), index=False)

pd.DataFrame({
    'y_true': y.values,
    'oof_mean': oof_mu,
    'oof_sigma': oof_sd,
    'broth_id': df['broth_id'].values,
    'diversity_level': df['diversity_level'].values,
    'community_size': df['community_size'].values,
}).to_csv(os.path.join(OUTDIR, f"xgbd03_v{VERSION}_oof_heldout_broth.csv"), index=False)

folds_df.to_csv(os.path.join(OUTDIR, f"xgbd03_v{VERSION}_groupkfold.csv"), index=False)
gss_df.to_csv(os.path.join(OUTDIR, f"xgbd03_v{VERSION}_groupshufflesplit.csv"), index=False)

# the bundle: everything a Bayesian optimizer needs, in one JSON
train_rows = df.loc[X_train.index]
r0_bounds = {}
for c in r0_cols:
    active_vals = train_rows.loc[train_rows[c] > 0, c]
    r0_bounds[c] = [float(active_vals.min()) if len(active_vals) else 0.0,
                    float(active_vals.max()) if len(active_vals) else 0.0]

total_r0 = train_rows[r0_cols].sum(axis=1)

bundle = {
    'model_family':        'xgboost_distribution',
    'version':             VERSION,
    'version_label':       version_label,
    'model_file':          os.path.basename(model_path),
    'loader':              'joblib',
    'feature_cols':        feature_cols,
    'r0_cols':             r0_cols,
    'active_cols':         active_cols,
    'extra_cols':          diversity_col,
    'quantiles':           None,
    'y_bounds':            [float(y.min()), float(y.max())],
    'r0_bounds':           r0_bounds,
    'total_r0_bounds':     [float(total_r0.min()), float(total_r0.max())],
    'total_r0_mean':       float(total_r0.mean()),
    'diversity_levels':    sorted(int(v) for v in df['diversity_level'].unique()),
    'sigma_scale':         sigma_scale,
    'calibration_level':   CALIB_LEVEL,
    'calibration_source':  'pooled_out_of_fold_heldout_broths',
    'metrics_interpolation': rep_interp,
    'metrics_heldout_broth': rep_oof,
    'best_params':         {k: (float(v) if isinstance(v, (int, float)) else v)
                            for k, v in best_params.items()},
}

bundle_path = os.path.join(OUTDIR, f"xgbd03_v{VERSION}_bundle.json")
with open(bundle_path, 'w') as f:
    json.dump(bundle, f, indent=2)

print(f"Model saved:       {model_path}")
print(f"Bundle saved:      {bundle_path}")
print(f"Predictions saved: xgbd03_v{VERSION}_predictions.csv")



Model saved:       ./xgbd03_v3_model.joblib
Bundle saved:      ./xgbd03_v3_bundle.json
Predictions saved: xgbd03_v3_predictions.csv


In [ ]:


# optimizer-readiness check
# The single interface every acquisition function needs. Reload from disk to
# prove the saved artefacts really are self-sufficient.

reloaded = joblib.load(model_path)
with open(bundle_path) as f:
    spec = json.load(f)


def predict_mean_std(X_new, model=reloaded, spec=spec, calibrated=True):
    """Return (mean, sigma) for a DataFrame or array of candidate nutrient rows.

    This is the exact signature a Bayesian optimization loop expects. Columns
    are reordered to the training order before predicting, because tree models
    index features positionally and a silent column reshuffle would corrupt
    every prediction without raising an error.
    """
    if isinstance(X_new, pd.DataFrame):
        Xq = X_new[spec['feature_cols']]
    else:
        Xq = pd.DataFrame(np.atleast_2d(X_new), columns=spec['feature_cols'])
    p = model.predict(Xq)
    sd = p.scale * (spec['sigma_scale'] if calibrated else 1.0)
    return np.asarray(p.loc), np.asarray(sd)


def upper_confidence_bound(X_new, kappa=2.0):
    """UCB acquisition: mean + kappa*sigma. Maximise this to pick the next broth."""
    mu, sd = predict_mean_std(X_new)
    return mu + kappa * sd


demo = X_test.iloc[:5]
mu_d, sd_d = predict_mean_std(demo)
ucb_d = upper_confidence_bound(demo, kappa=2.0)

print("\nOptimizer-readiness check (5 rows through the reloaded model):")
print(f"{'row':>4} {'mean':>9} {'sigma':>9} {'lo95':>9} {'hi95':>9} {'UCB(k=2)':>10} {'true':>9}")
for i in range(len(demo)):
    lo = mu_d[i] - norm.ppf(0.975) * sd_d[i]
    hi = mu_d[i] + norm.ppf(0.975) * sd_d[i]
    print(f"{i:>4} {mu_d[i]:>9.4f} {sd_d[i]:>9.4f} {lo:>9.4f} {hi:>9.4f} "
          f"{ucb_d[i]:>10.4f} {y_test.values[i]:>9.4f}")

assert np.allclose(mu_d, mu_final[:5], atol=1e-5), "reloaded model disagrees with in-memory model"
print("\nReloaded model reproduces in-memory predictions exactly.")


In [30]:



# summary

print(f"SUMMARY -- XGBDistribution VERSION {VERSION}: {version_label}")

print(f"Features:                    {len(feature_cols)}")
print(f"Trees used:                  {best_it}")
print(f"Interpolation R2:            {r2_final:.4f}")
print(f"Interpolation MAE:           {mae_final:.4f}")
print(f"Interpolation NLL:           {rep_interp['nll']:.4f}")
print(f"Interpolation CRPS:          {rep_interp['crps']:.4f}")
print(f"Interpolation 95% coverage:  {rep_interp['cov95']:.4f}")
print(f"Held-out-broth R2 (pooled):  {rep_oof['r2']:.4f}")
print(f"Held-out-broth NLL:          {rep_oof['nll']:.4f}")
print(f"Held-out-broth 95% coverage: {rep_oof['cov95']:.4f}")
print(f"Generalization gap (R2):     {r2_final - rep_oof['r2']:.4f}")
print(f"Sigma calibration factor:    {sigma_scale:.4f}")



SUMMARY -- XGBDistribution VERSION 3: Pure nutrients + diversity (no species, no size)
Features:                    41
Trees used:                  319
Interpolation R2:            0.6643
Interpolation MAE:           0.1394
Interpolation NLL:           -0.7005
Interpolation CRPS:          0.0979
Interpolation 95% coverage:  0.9489
Held-out-broth R2 (pooled):  0.4338
Held-out-broth NLL:          0.1688
Held-out-broth 95% coverage: 0.8943
Generalization gap (R2):     0.2305
Sigma calibration factor:    1.1991
